In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="5" color="black">ch7. 라이브러리를 이용한 시계열 데이터 분석</font>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

In [ ]:
# data.go.kr
df = pd.read_csv('data/일별_평균_대기오염도정보_2025.csv', encoding='cp949')
df.tail()

In [ ]:
# 특정 열의 결측치 확인
df[df['이산화질소농도(ppm)'].isna()].head()

In [ ]:
# 결측치가 존재하는 모든 열 확인
df[df.isna().any(axis=1)].head()

In [ ]:
# 모든 측정소명 결측치 수 
df.loc[df.isna().any(axis=1), '측정소명'].value_counts()

In [ ]:
# 결측치가 있는 측정소명 수 
len(df.loc[df.isna().any(axis=1), '측정소명'].value_counts())

In [ ]:
# 측정소명 확인
df['측정소명'].unique()

In [ ]:
# 측정소명 수 확인
df['측정소명'].nunique()

In [ ]:
# pd.to_datetime(df['측정일시']) - ms 단위로 인식
# 측정일시 컬럼 타입 변경 순서 : int → str → datetime
df['측정일시'] = pd.to_datetime(df['측정일시'].astype(str))

In [ ]:
df.info()

In [ ]:
df[df['측정소명']=='서울숲'].info()

In [ ]:
# 에러 발생 예시 : 결측치가 있는 데이터의 시계열 분석
loc_name = '관악구'
df_err = df[df['측정소명']==loc_name]
df_err.head()

In [ ]:
# 결측치가 없는 데이터의 시계열 분석
loc_name = '서울숲'
df_flt = df[df['측정소명']==loc_name]
df_flt.head()

In [ ]:
plt.figure(figsize=(25,6))
plt.plot(df_flt['측정일시'], df_flt['미세먼지농도(㎍/㎥)'])
plt.plot(df_flt['측정일시'], df_flt['초미세먼지농도(㎍/㎥)'])
plt.xlabel('측정일시')
plt.ylabel('먼지농도(㎍/㎥)')

# 1. statsmodels
- 데이터가 가진 트렌드, 계절성를 수학적으로 분리하여 주기적인 데이터의 특성 파악
- statsmodels 사용 시 데이터 전처리 : 중복 제거, 결측치 채우기, 빈도(간격) 설정
- 날짜 데이터를 반드시 인덱스로 지정

- seasonal_decomposed의 model
    - 1. 덧셈 모델 (model='additive')
$\text{실제값} = \text{추세} + \text{계절성} + \text{잔차}$
    - 2. 곱셈 모델 (model='multiplicative'): 
$\text{실제값} = \text{추세} \times \text{계절성} \times \text{잔차}$


In [ ]:
df_flt2 = df_flt[['측정일시', '미세먼지농도(㎍/㎥)']]
ts = df_flt2.set_index('측정일시')

In [ ]:
ts.info()

In [ ]:
# 시계열 데이터 주기성 분해
from statsmodels.tsa.seasonal import seasonal_decompose
rslt = seasonal_decompose(
    ts['미세먼지농도(㎍/㎥)'],
    period=30, # 주기
    model='additive' 
)

In [ ]:
# rslt.observed   # 실제값 : 원본 미세먼지 농도
# rslt.trend # 추이 : 계절/노이즈가 제거된 장기 변동 흐름
# rslt.seasonal # 계절성 : 30일 주기로 반복되는 패턴 수치
# rslt.resid  # 잔차 : 이벤트성/무작위 노이즈

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(25,12))
axes[0].plot(rslt.observed)
axes[1].plot(rslt.trend)
axes[2].plot(rslt.seasonal)
axes[3].plot(rslt.resid)
plt.tight_layout()
plt.show()


| 모델/라이브러리 | 용도 |
|---|---|
| seasonal_decompose | 통계적 추이 분석 |
| RNN / LSTM / GRU / Seq2Seq | 딥러닝 기반 예측 |
| Prophet | 추이 분석 및 미래 예측 |

# 2. Prophet
- Meta(구 Facebook)에서 개발한 오픈소스 시계열 예측 라이브러리
- 트렌드, 강력한 계절성, 그리고 휴일 효과가 결합된 시계열 데이터 분석 도구
- 누락된 날짜(결측치)가 있거나 불규칙한 간격의 데이터도 전처리 없이 강력하게 예측
- Prophet 사용 시 데이터 조건 : 데이터프레임의 컬럼 이름을 반드시 'ds(날짜 컬럼)', 'y(타겟 변수)'로 지정

In [ ]:
df_flt2.columns = ['ds','y']

In [ ]:
df_flt2.info()

In [ ]:
from prophet import Prophet

# 모델 생성
p_model = Prophet()

# 모델 학습
p_model.fit(df_flt2)

In [ ]:
# p_model을 이용하여 30일 이후의 데이터 예측 및 추이 분석

# 미래 데이터 프레임 생성 (30일 예측)
future = p_model.make_future_dataframe(periods=30)

# 예측 수행
forecast = p_model.predict(future)

# 시각화
p = p_model.plot(forecast)

In [ ]:
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'trend']].tail(30)

In [ ]:
# 트렌드와 휴일 효과 시각화
p = p_model.plot_components(forecast)

In [ ]:
forecast.loc[forecast['ds']=='2026-01-15', ['yhat', 'yhat_lower','yhat_upper']]